![Traffic](traffic.png)

Traffic data fluctuates constantly or is affected by time. Predicting it can be challenging, but this task will help sharpen your time-series skills. With deep learning, you can use abstract patterns in data that can help boost predictability.

Your task is to build a system that can be applied to help you predict traffic volume or the number of vehicles passing at a specific point and time. Determining this can help reduce road congestion, support new designs for roads or intersections, improve safety, and more! Or, you can use to help plan your commute to avoid traffic!

The dataset provided contains the hourly traffic volume on an interstate highway in Minnesota, USA. It also includes weather features and holidays, which often impact traffic volume.

Time to predict some traffic!

### The data:

The dataset is collected and maintained by UCI Machine Learning Repository. The target variable is `traffic_volume`. The dataset contains the following and has already been normalized and saved into training and test sets:

`train_scaled.csv`, `test_scaled.csv`
| Column     | Type       | Description              |
|------------|------------|--------------------------|
|`temp`                   |Numeric            |Average temp in kelvin|
|`rain_1h`                |Numeric            |Amount in mm of rain that occurred in the hour|
|`snow_1h`                |Numeric            |Amount in mm of snow that occurred in the hour|
|`clouds_all`             |Numeric            |Percentage of cloud cover|
|`date_time`              |DateTime           |Hour of the data collected in local CST time|
|`holiday_` (11 columns)  |Categorical        |US National holidays plus regional holiday, Minnesota State Fair|
|`weather_main_` (11 columns)|Categorical     |Short textual description of the current weather|
|`weather_description_` (35 columns)|Categorical|Longer textual description of the current weather|
|`hour_of_day`|Numeric|The hour of the day|
|`day_of_week`|Numeric|The day of the week (0=Monday, Sunday=6)|
|`day_of_month`|Numeric|The day of the month|
|`month`|Numeric|The number of the month|
|`traffic_volume`         |Numeric            |Hourly I-94 ATR 301 reported westbound traffic volume|

In [128]:
# Import the relevant libraries
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [129]:
# Read the traffic data from the CSV training and test files
train_scaled_df = pd.read_csv('train_scaled.csv')
test_scaled_df = pd.read_csv('test_scaled.csv')

# Convert the DataFrame to NumPy arrays
train_scaled = train_scaled_df.to_numpy()
test_scaled = test_scaled_df.to_numpy()

In [130]:
# Start coding here
# Use as many cells as you like!
train_scaled_df.head()

,temp,rain_1h,snow_1h,clouds_all,holiday_Christmas Day,holiday_Columbus Day,holiday_Independence Day,holiday_Labor Day,holiday_Martin Luther King Jr Day,holiday_Memorial Day,holiday_New Years Day,holiday_State Fair,holiday_Thanksgiving Day,holiday_Veterans Day,holiday_Washingtons Birthday,weather_main_Clear,weather_main_Clouds,weather_main_Drizzle,weather_main_Fog,weather_main_Haze,weather_main_Mist,weather_main_Rain,weather_main_Smoke,weather_main_Snow,weather_main_Squall,weather_main_Thunderstorm,weather_description_SQUALLS,weather_description_Sky is Clear,weather_description_broken clouds,weather_description_drizzle,weather_description_few clouds,weather_description_fog,weather_description_haze,weather_description_heavy intensity drizzle,weather_description_heavy intensity rain,weather_description_heavy snow,weather_description_light intensity drizzle,weather_description_light intensity shower rain,weather_description_light rain,weather_description_light rain and snow,weather_description_light shower snow,weather_description_light snow,weather_description_mist,weather_description_moderate rain,weather_description_overcast clouds,weather_description_proximity shower rain,weather_description_proximity thunderstorm,weather_description_proximity thunderstorm with drizzle,weather_description_proximity thunderstorm with rain,weather_description_scattered clouds,weather_description_shower drizzle,weather_description_shower snow,weather_description_sky is clear,weather_description_smoke,weather_description_snow,weather_description_thunderstorm,weather_description_thunderstorm with heavy rain,weather_description_thunderstorm with light drizzle,weather_description_thunderstorm with light rain,weather_description_thunderstorm with rain,weather_description_very heavy rain,hour_of_day,day_of_week,day_of_month,month,traffic_volume
0,0.935245,0.0,0.0,0.40,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.391304,0.166667,0.033333,0.818182,0.761676
1,0.938749,0.0,0.0,0.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.434783,0.166667,0.033333,0.818182,0.620330
2,0.939463,0.0,0.0,0.90,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.478261,0.166667,0.033333,0.818182,0.654808
3,0.941247,0.0,0.0,0.90,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.521739,0.166667,0.033333,0.818182,0.690385
4,0.944524,0.0,0.0,0.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.565217,0.166667,0.033333,0.818182,0.675549


In [131]:
train_scaled_df.iloc[0, -1]

0.7616758241758241

In [132]:
len(train_scaled)
train_scaled.shape

(34042, 66)

In [133]:
def create_sequences(input_df, sequence_length, target_index):
    Xs, ys = [], []
    for i in range(len(input_df) - sequence_length):
        x = input_df.iloc[i:i+sequence_length,:]
        y = input_df.iloc[i+sequence_length, target_index]

        Xs.append(x)
        ys.append(y)

    return np.array(Xs), np.array(ys)          

In [134]:
X_train, y_train = create_sequences(train_scaled_df, 80, -1) 
X_test, y_test = create_sequences(test_scaled_df, 80, -1)

#### Check resutls of create_sequences

In [135]:
y_train[:5]

array([0.25164835, 0.16909341, 0.09862637, 0.07486264, 0.05645604])

In [136]:
X_train[0].shape

(80, 66)

In [137]:
train_dataset = TensorDataset(
    torch.from_numpy(X_train.astype(np.float32)).float(),
    torch.from_numpy(y_train.astype(np.float32)).float()
)

test_dataset = TensorDataset(
    torch.from_numpy(X_test.astype(np.float)).float(),
    torch.from_numpy(y_test.astype(np.float)).float()
)

print(f"train_dataset = {train_dataset}")
print(f"test_dataset  = {test_dataset}")

train_dataset = <torch.utils.data.dataset.TensorDataset object at 0x7f665c53f220>
test_dataset  = <torch.utils.data.dataset.TensorDataset object at 0x7f665c53fe20>


In [138]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

train_X_example, train_y_example = next(iter(train_loader))
train_X_example.shape

torch.Size([64, 80, 66])

### define the neural network
* create a two layer lstm RNN
* define input_size as the input vector dimension (66 here)
* define output_size for RNN, we use 32
* define linear layer to convert 32 vector to 1 as the output prediction for traffic volume

In [139]:
class Net(nn.Module):
    def __init__(self, input_size: int):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = 32
        self.lstm = nn.LSTM(input_size=self.input_size,
                           hidden_size=self.hidden_size,
                           num_layers=2,
                           batch_first=True)
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), self.hidden_size)
        c0 = torch.zeros(2, x.size(0), self.hidden_size)
        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out[:, -1, :])

#### Test lstm network
* get one training example and test it outputs the correct shaped tensor

In [140]:
traffic_model = Net(66)
rs = traffic_model(train_X_example).squeeze(dim=1)
rs.shape

torch.Size([64])

In [141]:
train_y_example.unsqueeze(dim=1).shape

torch.Size([64, 1])

In [142]:
loss_fun = nn.MSELoss()
optimizer = optim.Adam(lstm_net.parameters(), lr=0.001)
epoches = 2
train_loss = 0
train_losses = []

for ep in range(epoches):
    train_losses = []
    test_losses = []

    traffic_model.train()

    for X, y in train_loader:
        optimizer.zero_grad()
        pred_y = traffic_model(X)
        train_loss = loss_fun(y, pred_y)

        train_losses.append(train_loss.item())
        train_loss.backward()
        optimizer.step()
    print(f"epoch{ep} training loss is {sum(train_losses)/len(train_losses)}")

    traffic_model.eval()
    with torch.no_grad():    
        for test_X, test_y in test_loader:
            pred_y = traffic_model(test_X)
            test_loss = loss_fun(test_y, pred_y)
    
            test_losses.append(test_loss.item())
    print(f"epoch{ep} test loss is {sum(test_losses)/len(test_losses)}")    


        
    

epoch0 training loss is 0.26787671472056435
epoch0 test loss is 0.271566735857194
epoch1 training loss is 0.26780673796959964
epoch1 test loss is 0.271566735857194


In [143]:
final_training_loss = train_loss.item()
final_training_loss

0.21087808907032013

In [144]:
test_labels, test_preds = [], []

traffic_model.eval()
with torch.no_grad():    
    for test_X, test_y in test_loader:
        pred_y = traffic_model(test_X).squeeze(dim=1)
        test_labels.append(test_y)
        test_preds.append(pred_y)

In [145]:
test_labels_final = torch.cat(test_labels)
print(f"test_labels shape = {test_labels_final.shape}")

test_preds_final = torch.cat(test_preds)
test_preds_final.shape
print(f"test_preds shape = {test_preds_final.shape}")

test_labels shape = torch.Size([6453])
test_preds shape = torch.Size([6453])


In [146]:
test_mse = F.mse_loss(test_labels_final, test_preds_final)
print(test_mse)

tensor(0.2695)
